# Feature Engineering and Target Construction

This notebook builds the processed modeling datasets used by the final modeling notebook. Step 1 creates a clean ticker-hour table. Step 2 creates the hybrid intraday/overnight target with a tunable return threshold.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import polars as pl
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import (
    CLEAN_FINANCE_START,
    HYBRID_TARGET_DATASET_PATH,
    MAX_INTRADAY_HORIZON_HOURS,
    MAX_OVERNIGHT_HORIZON_HOURS,
    MODELING_DATASET_PATH,
    PROCESSED_DATA_DIR,
    TARGET_RETURN_THRESHOLD,
)
from src.data_loading import load_merged_data
from src.feature_engineering import (
    add_next_close_target,
    build_clean_hourly_dataset,
    build_hybrid_target_dataset,
)
from src.plots import set_plot_style

set_plot_style()
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

df = load_merged_data()
display(df.head())


Ticker,Timestamp,Text,Sentiment,Post_Count,Open,High,Low,Close,Volume
str,"datetime[μs, UTC]",str,f64,i64,f64,f64,f64,f64,i64
"""$AAPL""",2025-07-07 23:00:00 UTC,"""BREAKING: Apple’s, $AAPL, top …",-0.220868,2,210.130005,210.490005,208.449997,209.279999,9381795
"""$AAPL""",2025-07-07 23:00:00 UTC,"""🔴 Importance: 65/100 #news #To…",-0.440318,2,210.130005,210.490005,208.449997,209.279999,9381795
"""$AAPL""",2025-07-08 00:00:00 UTC,"""What do you think #investors? …",0.150104,2,210.130005,210.490005,208.449997,209.279999,9381795
"""$AAPL""",2025-07-08 00:00:00 UTC,"""Lemme guess... you bought AAPL…",0.012221,2,210.130005,210.490005,208.449997,209.279999,9381795
"""$AAPL""",2025-07-08 02:00:00 UTC,"""$AAPL Support / Resistance Lev…",0.027583,1,210.130005,210.490005,208.449997,209.279999,9381795


## Baseline Target Audit

This is the target used in the original notebook. We keep this audit cell to show why the post-level baseline target is not the final modeling target.

In [2]:
baseline_df = add_next_close_target(df)

print("Post-level target balance:")
print(baseline_df["Target_Direction"].value_counts().sort("Target_Direction"))


Post-level target balance:
shape: (2, 2)
┌──────────────────┬────────┐
│ Target_Direction ┆ count  │
│ ---              ┆ ---    │
│ i32              ┆ u32    │
╞══════════════════╪════════╡
│ 0                ┆ 187033 │
│ 1                ┆ 7835   │
└──────────────────┴────────┘


## Step 1: Clean Ticker-Hour Modeling Input

The raw dataset has one row per post. For final modeling, one row per ticker-hour is cleaner because every post in the same ticker-hour shares the same market price target.

This step also filters out old social posts before the reliable Yahoo Finance hourly coverage window. The current cutoff is `2024-06-01`.

In [3]:
clean_hourly_df = build_clean_hourly_dataset(df)

print(f"Finance coverage cutoff: {CLEAN_FINANCE_START:%Y-%m-%d}")
print(f"Raw rows: {df.height:,}")
print(f"Clean ticker-hour rows: {clean_hourly_df.height:,}")
print(f"Unique tickers: {clean_hourly_df.select('Ticker').n_unique()}")
display(clean_hourly_df.head())


Finance coverage cutoff: 2024-06-01
Raw rows: 194,891
Clean ticker-hour rows: 54,572
Unique tickers: 23


Ticker,Timestamp,Open,High,Low,Close,Volume,Sentiment_Mean,Sentiment_Median,Sentiment_Std,Sentiment_Min,Sentiment_Max,Positive_Post_Count,Negative_Post_Count,Neutral_Post_Count,Post_Count,Bullishness_Index
str,"datetime[μs, UTC]",f64,f64,f64,f64,i64,f64,f64,f64,f64,f64,i64,i64,i64,u32,f64
"""$AAPL""",2025-07-07 23:00:00 UTC,210.130005,210.490005,208.449997,209.279999,9381795,-0.330593,-0.330593,0.155175,-0.440318,-0.220868,0,2,0,2,-1.0
"""$AAPL""",2025-07-08 00:00:00 UTC,210.130005,210.490005,208.449997,209.279999,9381795,0.081162,0.081162,0.097498,0.012221,0.150104,1,0,1,2,0.5
"""$AAPL""",2025-07-08 02:00:00 UTC,210.130005,210.490005,208.449997,209.279999,9381795,0.027583,0.027583,0.0,0.027583,0.027583,0,0,1,1,0.0
"""$AAPL""",2025-07-08 05:00:00 UTC,210.130005,210.490005,208.449997,209.279999,9381795,-0.033409,0.075754,0.479426,-0.558003,0.382022,2,1,0,3,0.333333
"""$AAPL""",2025-07-08 06:00:00 UTC,210.130005,210.490005,208.449997,209.279999,9381795,0.019368,0.019368,0.0,0.019368,0.019368,0,0,1,1,0.0


## Step 1 Validation

In [4]:
duplicate_ticker_hours = clean_hourly_df.height - clean_hourly_df.select(["Ticker", "Timestamp"]).unique().height

print(f"Duplicate ticker-hour rows: {duplicate_ticker_hours}")
print("Timestamp range:")
display(clean_hourly_df.select(pl.col("Timestamp").min().alias("min_ts"), pl.col("Timestamp").max().alias("max_ts")))

print("Null counts:")
print(clean_hourly_df.null_count())

print("Bullishness index bounds:")
display(clean_hourly_df.select(pl.col("Bullishness_Index").min().alias("min"), pl.col("Bullishness_Index").max().alias("max")))

print("Ticker-hour rows by ticker:")
display(clean_hourly_df.group_by("Ticker").len().sort("len", descending=True))


Duplicate ticker-hour rows: 0
Timestamp range:


min_ts,max_ts
"datetime[μs, UTC]","datetime[μs, UTC]"
2024-06-01 15:00:00 UTC,2026-04-28 23:00:00 UTC


Null counts:
shape: (1, 17)
┌────────┬───────────┬──────┬──────┬───┬───────────────┬───────────────┬────────────┬──────────────┐
│ Ticker ┆ Timestamp ┆ Open ┆ High ┆ … ┆ Negative_Post ┆ Neutral_Post_ ┆ Post_Count ┆ Bullishness_ │
│ ---    ┆ ---       ┆ ---  ┆ ---  ┆   ┆ _Count        ┆ Count         ┆ ---        ┆ Index        │
│ u32    ┆ u32       ┆ u32  ┆ u32  ┆   ┆ ---           ┆ ---           ┆ u32        ┆ ---          │
│        ┆           ┆      ┆      ┆   ┆ u32           ┆ u32           ┆            ┆ u32          │
╞════════╪═══════════╪══════╪══════╪═══╪═══════════════╪═══════════════╪════════════╪══════════════╡
│ 0      ┆ 0         ┆ 0    ┆ 0    ┆ … ┆ 0             ┆ 0             ┆ 0          ┆ 0            │
└────────┴───────────┴──────┴──────┴───┴───────────────┴───────────────┴────────────┴──────────────┘
Bullishness index bounds:


min,max
f64,f64
-1.0,1.0


Ticker-hour rows by ticker:


Ticker,len
str,u32
"""$PLTR""",5055
"""$MSTR""",4823
"""$GOOGL""",4530
"""$GME""",4522
"""$MSFT""",4190
…,…
"""$HOOD""",378
"""$COIN""",337
"""$SPY""",324


## Save Step 1 Modeling Input

This table is the clean targetless modeling input. The target-bearing dataset is built in Step 2.

In [5]:
clean_hourly_df.write_csv(MODELING_DATASET_PATH)
print(f"Saved {clean_hourly_df.height:,} rows to {MODELING_DATASET_PATH}")


Saved 54,572 rows to /home/laytc/GitHubRepo/Penn/cis2450/cis-2450-final-proj/data/processed/modeling_dataset.csv


## Step 2: Hybrid Intraday/Overnight Target

The final target is thresholded and tunable:

- `Target_Direction = 1` when future return is greater than `TARGET_RETURN_THRESHOLD`.
- `Target_Direction = 0` when future return is less than `-TARGET_RETURN_THRESHOLD`.
- Tiny moves between those thresholds are dropped as noisy neutral moves.

Hybrid target logic:

- **Intraday rows** use regular-session sentiment at hour `t` to predict the next observed same-day market bar.
- **Overnight rows** aggregate off-market sentiment before the next observed market open and predict the gap from the previous observed market close to that open.

In [6]:
hybrid_df = build_hybrid_target_dataset(
    clean_hourly_df,
    threshold=TARGET_RETURN_THRESHOLD,
    max_intraday_horizon_hours=MAX_INTRADAY_HORIZON_HOURS,
    max_overnight_horizon_hours=MAX_OVERNIGHT_HORIZON_HOURS,
    drop_neutral=True,
)

print(f"Return threshold: +/-{TARGET_RETURN_THRESHOLD:.4%}")
print(f"Max intraday target horizon: {MAX_INTRADAY_HORIZON_HOURS} hours")
print(f"Max overnight target horizon: {MAX_OVERNIGHT_HORIZON_HOURS} hours")
print(f"Hybrid target rows: {hybrid_df.height:,}")
display(hybrid_df.head())


Return threshold: +/-0.1000%
Max intraday target horizon: 6 hours
Max overnight target horizon: 96 hours
Hybrid target rows: 13,325


Target_Type,Ticker,Timestamp,Signal_Start,Signal_End,Target_Timestamp,Reference_Timestamp,Target_Horizon_Hours,Open,High,Low,Close,Volume,Sentiment_Mean,Sentiment_Median,Sentiment_Std,Sentiment_Min,Sentiment_Max,Positive_Post_Count,Negative_Post_Count,Neutral_Post_Count,Post_Count,Bullishness_Index,Reference_Price,Target_Price,Target_Return,Target_Direction
str,str,"datetime[μs, UTC]","datetime[μs, UTC]","datetime[μs, UTC]","datetime[μs, UTC]","datetime[μs, UTC]",f64,f64,f64,f64,f64,i64,f64,f64,f64,f64,f64,i64,i64,i64,i64,f64,f64,f64,f64,i8
"""intraday""","""$AAPL""",2025-07-08 13:00:00 UTC,2025-07-08 13:00:00 UTC,2025-07-08 13:00:00 UTC,2025-07-08 14:00:00 UTC,2025-07-08 13:00:00 UTC,1.0,210.130005,210.490005,208.449997,209.279999,9381795,-0.218546,-0.006071,0.411388,-0.692725,0.043158,0,1,2,3,-0.333333,209.279999,209.870697,0.002823,1
"""overnight""","""$AAPL""",2025-07-09 12:00:00 UTC,2025-07-08 20:00:00 UTC,2025-07-09 12:00:00 UTC,2025-07-09 13:00:00 UTC,2025-07-08 19:00:00 UTC,1.0,210.035004,210.035004,210.035004,210.035004,5121140,0.043206,0.028212,0.402097,-0.937532,0.923812,9,3,5,17,0.352941,210.035004,209.529999,-0.002404,0
"""intraday""","""$AAPL""",2025-07-09 13:00:00 UTC,2025-07-09 13:00:00 UTC,2025-07-09 13:00:00 UTC,2025-07-09 14:00:00 UTC,2025-07-09 13:00:00 UTC,1.0,209.529999,210.899994,209.350006,209.910004,8992949,-0.367082,-0.304557,0.439264,-0.852405,0.045089,0,3,3,6,-0.5,209.910004,207.804993,-0.010028,0
"""intraday""","""$AAPL""",2025-07-09 15:00:00 UTC,2025-07-09 15:00:00 UTC,2025-07-09 15:00:00 UTC,2025-07-09 16:00:00 UTC,2025-07-09 15:00:00 UTC,1.0,207.809998,208.220001,207.2659,207.800003,4676493,0.397451,0.254352,0.477901,0.007448,0.930554,2,0,1,3,0.666667,207.800003,208.915604,0.005369,1
"""intraday""","""$AAPL""",2025-07-09 16:00:00 UTC,2025-07-09 16:00:00 UTC,2025-07-09 16:00:00 UTC,2025-07-09 17:00:00 UTC,2025-07-09 16:00:00 UTC,1.0,207.810394,209.044998,207.571396,208.915604,3964765,0.030129,0.030129,0.020721,0.015476,0.044781,0,0,2,2,0.0,208.915604,209.732498,0.00391,1


## Step 2 Validation

In [7]:
print("Rows by target type and direction:")
display(hybrid_df.group_by(["Target_Type", "Target_Direction"]).len().sort(["Target_Type", "Target_Direction"]))

print("Rows by target type:")
display(hybrid_df.group_by("Target_Type").len().sort("Target_Type"))

print("Return summary:")
display(
    hybrid_df.group_by("Target_Type").agg([
        pl.col("Target_Return").min().alias("min"),
        pl.col("Target_Return").quantile(0.25).alias("p25"),
        pl.col("Target_Return").median().alias("median"),
        pl.col("Target_Return").quantile(0.75).alias("p75"),
        pl.col("Target_Return").max().alias("max"),
    ]).sort("Target_Type")
)

print("Target horizon summary:")
display(
    hybrid_df.group_by("Target_Type").agg([
        pl.col("Target_Horizon_Hours").min().alias("min_h"),
        pl.col("Target_Horizon_Hours").median().alias("median_h"),
        pl.col("Target_Horizon_Hours").max().alias("max_h"),
    ]).sort("Target_Type")
)

print("Null total:", hybrid_df.null_count().sum_horizontal().item())
print("Target date range:")
display(hybrid_df.select(pl.col("Timestamp").min().alias("min_signal_ts"), pl.col("Target_Timestamp").max().alias("max_target_ts")))


Rows by target type and direction:


Target_Type,Target_Direction,len
str,i8,u32
"""intraday""",0,4995
"""intraday""",1,5001
"""overnight""",0,1587
"""overnight""",1,1742


Rows by target type:


Target_Type,len
str,u32
"""intraday""",9996
"""overnight""",3329


Return summary:


Target_Type,min,p25,median,p75,max
str,f64,f64,f64,f64,f64
"""intraday""",-0.223698,-0.004648,0.001006,0.004695,0.200241
"""overnight""",-0.332619,-0.00978,0.001671,0.011567,0.445334


Target horizon summary:


Target_Type,min_h,median_h,max_h
str,f64,f64,f64
"""intraday""",1.0,1.0,6.0
"""overnight""",1.0,2.0,88.0


Null total: 0
Target date range:


min_signal_ts,max_target_ts
"datetime[μs, UTC]","datetime[μs, UTC]"
2024-06-03 13:00:00 UTC,2026-04-28 19:00:00 UTC


## Save Step 2 Target Dataset

In [8]:
hybrid_df.write_csv(HYBRID_TARGET_DATASET_PATH)
print(f"Saved {hybrid_df.height:,} rows to {HYBRID_TARGET_DATASET_PATH}")


Saved 13,325 rows to /home/laytc/GitHubRepo/Penn/cis2450/cis-2450-final-proj/data/processed/hybrid_target_dataset.csv


## Next Feature Engineering Work

- Add lag returns, rolling returns, volume anomaly, sentiment EMA, sentiment z-score, hour/day features, ticker encoding, and `Target_Type` encoding.
- Use `data/processed/hybrid_target_dataset.csv` as the modeling notebook input.
- Apply any scaling or sampling only after the train/test split in the modeling notebook.